[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SotaYoshida/book_NumericalAnalysis/blob/main/notebooks/Chap_DiffusionEquation.ipynb)

# 拡散方程式の数値解法

第9章では、時刻によって変わる状態を追った。棒の温度では、同じ時刻でも場所ごとに値が違うため、$u(t)$ を $u(x,t)$ へ広げる。偏微分方程式（partial differential equation; PDE）は、このような複数の変数に依存する未知関数の偏微分を含む方程式である。本章では、棒の温度や物質の濃度が拡散する過程を表す1次元拡散方程式

$$
u_t=\alpha u_{xx},\qquad 0<x<1,\quad t>0,
$$

を題材にする。$u_t$ は位置を固定した時間変化率、$u_{xx}$ は時刻を固定した空間方向の2階微分を表す。$u(x,t)$ は位置 $x$、時刻 $t$ における温度など、$\alpha>0$ は拡散係数である。

PDEを計算機で解くには、連続な空間と時間を有限個の値で表さなければならない。本章では

$$
\text{PDE}
\longrightarrow \text{空間差分による連立ODE}
\longrightarrow \text{時間積分}
\longrightarrow \text{疎な線形代数}
$$

という流れをたどる。これは、第8章の差分、第9章のODE、第5章の連立一次方程式を一つの問題の中で組み合わせる例である。

## この章の目標

- PDE、初期条件、境界条件がそれぞれ何を指定するか説明できる。
- 2階中心差分を導出し、空間の離散化誤差を説明できる。
- 半離散化を連立ODEおよび疎行列の形で表せる。
- 陽Euler法の更新を近傍値の平均として読み、刻み幅の制約を説明できる。
- 後退Euler法を線形方程式へ変形し、行列分解を再利用できる。
- 格子を細かくした結果を比べ、空間と時間の両方に誤差があることを説明できる。


## PDEだけでは解は定まらない

時間発展を始めるには、初期時刻の空間分布

$$
u(x,0)=u_0(x)
$$

が必要である。また有限区間では境界 $x=0,1$ での振る舞いも指定する。本章の基本例では

$$
u(0,t)=u(1,t)=0
$$

というDirichlet（ディリクレ）境界条件を使う。これは棒の両端を温度0に固定する状況に対応する。

別の代表例であるNeumann（ノイマン）境界条件 $u_x=0$ は、境界から熱や物質が流出しない断熱・無流束条件に対応する。同じPDEでも境界条件が変われば解の長時間挙動は変わる。Dirichlet条件では熱が境界から逃げて最終的に0へ近づく一方、両端が無流束なら空間平均が保存される。

初期値と境界値は両立していることが望ましい。本章の解析解による試験には

$$
u_0(x)=\sin(\pi x),\qquad
u(x,t)=e^{-\alpha\pi^2t}\sin(\pi x)
$$

を用いる。この初期値は両端で0になり、境界条件を満たす。


## 空間を格子にし、2階微分を差分にする

区間 $[0,1]$ を $m$ 等分し、

$$
x_j=j\Delta x,\qquad \Delta x=\frac1m,
\qquad j=0,1,\ldots,m
$$

とする。$u_j(t)\approx u(x_j,t)$ と置く。

$x_j$ の左右でTaylor展開すると、

$$
\begin{aligned}
u(x_j+\Delta x,t)&=u+\Delta x\,u_x
+\frac{\Delta x^2}{2}u_{xx}
+\frac{\Delta x^3}{6}u_{xxx}+O(\Delta x^4),\\
u(x_j-\Delta x,t)&=u-\Delta x\,u_x
+\frac{\Delta x^2}{2}u_{xx}
-\frac{\Delta x^3}{6}u_{xxx}+O(\Delta x^4).
\end{aligned}
$$

二式を足して整理すれば、

$$
u_{xx}(x_j,t)=
\frac{u(x_{j-1},t)-2u(x_j,t)+u(x_{j+1},t)}{\Delta x^2}
+O(\Delta x^2)
$$

を得る。これは第8章の2階中心差分であり、空間について2次精度である。分子は「両隣の平均と中心値との差」を測る。中心が周囲より高ければ負、低ければ正となるため、拡散が凹凸を平らにする向きと整合する。


## 空間半離散化：PDEから連立ODEへ

差分公式をPDEへ代入すると、内部格子点 $j=1,\ldots,m-1$ で

$$
\frac{du_j}{dt}=
\frac{\alpha}{\Delta x^2}
(u_{j-1}-2u_j+u_{j+1})
$$

を得る。空間だけを離散化し、時間をまだ連続のまま残す方法を**method of lines**という。

まず内部点が3個だけなら、境界を含めた点は $j=0,1,2,3,4$ の5個である。端の値 $u_0=u_4=0$ を代入すると、未知数は $u_1,u_2,u_3$ のみで、

$$
\frac{d}{dt}\begin{bmatrix}u_1\\u_2\\u_3\end{bmatrix}
=\frac{\alpha}{\Delta x^2}
\begin{bmatrix}-2&1&0\\1&-2&1\\0&1&-2\end{bmatrix}
\begin{bmatrix}u_1\\u_2\\u_3\end{bmatrix}
$$

となる。最初と最後の行に未知の境界値がないのは、値が既に指定されているためである。

一般に、内部値を

$$
v(t)=[u_1(t),u_2(t),\ldots,u_{m-1}(t)]^T
$$

と並べると、境界値が0の場合は

$$
v'(t)=\frac{\alpha}{\Delta x^2}Lv(t),\qquad
L=\begin{bmatrix}
-2&1&&\\
1&-2&1&\\
&\ddots&\ddots&\ddots\\
&&1&-2
\end{bmatrix}
$$

となる。$L$ は対称三重対角行列で、各行の非零要素は高々3個である。未知数が増えても非零要素数は $O(m)$ にしか増えない。この構造を保存する疎行列を使うことが、大規模PDE計算の第一歩である。


In [1]:
using LinearAlgebra, SparseArrays

function laplacian_1d(n)
    n ≥ 1 || error("内部格子点数は正にしてください")
    return spdiagm(-1 => ones(n-1), 0 => fill(-2.0, n), 1 => ones(n-1))
end

for n in (9, 99, 999)
    Ltest = laplacian_1d(n)
    println("大きさ=$(size(Ltest)), 非零要素数=$(nnz(Ltest)), 密行列なら要素数=$(n^2)")
end


大きさ=(9, 9), 非零要素数=25, 密行列なら要素数=81
大きさ=(99, 99), 非零要素数=295, 密行列なら要素数=9801
大きさ=(999, 999), 非零要素数=2995, 密行列なら要素数=998001


## 陽Euler法：差分値から次の時刻を直接計算する

連立ODEへ前進Euler法を適用すると、

$$
u_j^{n+1}=u_j^n+
\mu(u_{j-1}^n-2u_j^n+u_{j+1}^n),
\qquad
\mu=\frac{\alpha\Delta t}{\Delta x^2}
$$

を得る。例えば近傍3点が $(1,2,1)$、$\mu=0.4$ なら、中心の新しい値は $0.4\times1+0.2\times2+0.4\times1=1.2$ となり、高い部分が下がる。添字の上付き $n$ は時刻、下付き $j$ は空間位置を表す。更新式を

$$
u_j^{n+1}=\mu u_{j-1}^n+(1-2\mu)u_j^n+\mu u_{j+1}^n
$$

と書くと、$0\leq\mu\leq1/2$ のとき三つの係数は非負で、和は1である。したがって新しい値は古い時刻の近傍値の重み付き平均になり、拡散らしく凹凸をならす。$\mu>1/2$ では中心の係数が負になり、高周波の誤差が増幅され得る。



```{figure} ../pic/learning_diffusion_stencil.svg
:name: fig-learning-diffusion-stencil
:width: 95%
:alt: 古い時刻の左1、中心2、右1から重み0.4、0.2、0.4で新しい中心値1.2を作る。

上段の3点はすべて時刻 $n$ の値で、下段の1点が時刻 $n+1$ の値である。左右の格子点も同様に更新するため、古い配列を読み、新しい配列へ書く。更新した値をすぐ隣の計算へ使うと、この式とは別の計算になる。
```

この条件は

$$
\Delta t\leq\frac{\Delta x^2}{2\alpha}
$$

を意味する。空間刻みを半分にすると、同じ安定条件を保つ時間刻みの上限は $1/4$ になる。ここでは格子幅によらず使える十分条件 $\mu\leq1/2$ を採用する。


In [2]:
function heat_explicit_steps(u0, α, dx, dt, nsteps)
    α > 0 && dx > 0 && dt ≥ 0 && nsteps ≥ 0 || error("刻みと係数を確認してください")
    u = float.(copy(u0))
    μ = α*dt/dx^2
    u[1] = u[end] = 0.0
    for _ in 1:nsteps
        unew = copy(u)
        for j in 2:length(u)-1
            unew[j] = u[j]+μ*(u[j-1]-2u[j]+u[j+1])
        end
        unew[1] = unew[end] = 0.0
        u = unew
    end
    return u
end

α = 1.0
nx = 41
x = collect(range(0.0, 1.0, length=nx))
dx = x[2]-x[1]
u0 = sin.(π*x)
μ = 0.4
dt = μ*dx^2/α

for nsteps in (0, 20, 100)
    solution_at_step = heat_explicit_steps(u0, α, dx, dt, nsteps)
    println("step=$nsteps: max(u)=$(round(maximum(solution_at_step),digits=6)), " *
            "境界値=($(solution_at_step[1]), $(solution_at_step[end]))")
end


step=0: max(u)=1.0, 境界値=(0.0, 0.0)
step=20: max(u)=0.951816, 境界値=(0.0, 0.0)
step=100: max(u)=0.781205, 境界値=(0.0, 0.0)


## 安定条件を実験で確かめる

不安定性は、滑らかな初期値だけではすぐ見えないことがある。丸め誤差や測定誤差に含まれる格子幅程度の細かな振動は、物理解の主要部分が減衰する間に増幅される。そのため、安定性試験では滑らかな初期値に微小な高周波成分を加えると現象を観察しやすい。

計算結果の見た目だけで安定性を判断してはいけない。短時間では自然に見えることがあり、また不安定な成分の初期振幅によって破綻が見える時刻が変わる。無次元量 $\mu$ を計算し、理論条件と照合する。


In [3]:
high_frequency = [(-1.0)^j for j in 0:nx-1]
high_frequency[1] = high_frequency[end] = 0.0
u0_test = u0+1e-8high_frequency

for μtest in (0.4, 0.6)
    dttest = μtest*dx^2/α
    utest = heat_explicit_steps(u0_test, α, dx, dttest, 100)
    println("μ=$μtest: max|u|=$(maximum(abs.(utest)))")
end


μ=0.4: max|u|=0.7812048334068494
μ=0.6: max|u|=3.848413293537218e6


In [4]:
# 図の再作成用（CairoMakieが必要）
using CairoMakie
CairoMakie.activate!(type="svg")
figure_dir = isdir("pic") ? "pic" : joinpath("..", "pic")
let
    fig = Figure(size=(1050,430), fontsize=16)
    ax = Axis(fig[1,1]; xlabel="position x", ylabel="u", title="Diffusion with fixed zero boundaries")
    for (steps, color) in zip((0,20,100), (:black,:orange,:dodgerblue))
        values = heat_explicit_steps(u0, α, dx, 0.4dx^2/α, steps)
        lines!(ax,x,values; color, linewidth=2.5,label="step $steps")
    end
    axislegend(ax;position=:rt)
    ax2 = Axis(fig[1,2];xlabel="time step",ylabel="perturbation norm / initial norm",
        yscale=log10, yticks=(10.0 .^ (-4:4:8), ["10⁻⁴", "10⁰", "10⁴", "10⁸"]),
        title="Small spatial oscillations")
    for (mu,color) in zip((0.4,0.6),(:dodgerblue,:tomato))
        amplitudes = [norm(heat_explicit_steps(high_frequency,α,dx,mu*dx^2/α,k))/norm(high_frequency) for k in 0:60]
        lines!(ax2,0:60,amplitudes;color,linewidth=2.5,label="mu = $mu")
    end
    axislegend(ax2;position=:lt)
    save(joinpath(figure_dir,"pde_diffusion_stability.svg"),fig)
end


```{figure} ../pic/pde_diffusion_stability.svg
:name: fig-pde-diffusion-stability
:width: 90%
:alt: 両端を0に固定した拡散方程式の平滑化と、細かな空間振動の増幅の比較

本文と同じ格子・境界条件を使う。左は $\mu=0.4$ で初期分布 $\sin(\pi x)$ が減衰する様子。右は、交互に符号が変わる摂動だけを時間発展させ、初期ノルムで割った値である。線形な方程式なので、微小な摂動にも同じ倍率が働く。$\mu=0.6$ では、この成分が増幅してしまう。
```


## 解析解による格子細分化試験

$u_0(x)=\sin(\pi x)$ に対する解析解

$$
u(x,t)=e^{-\alpha\pi^2t}\sin(\pi x)
$$

を使い、空間格子を細かくしたときの誤差を調べる。ただし陽Euler法では $\Delta t$ と $\Delta x$ を独立に選べず、安定条件のため $\Delta t=O(\Delta x^2)$ とする必要がある。この選択では空間差分の $O(\Delta x^2)$ と時間Euler法の $O(\Delta t)$ がどちらも $O(\Delta x^2)$ になり、全体として2次に近い収束が予想される。

最終時刻 $T$ を固定するには、まずstep数を整数へ丸め、その後 $\Delta t=T/N_t$ と再計算する。そうしないと格子ごとに異なる時刻の解を比較してしまう。


In [5]:
function explicit_to_time(u0, α, dx, dt_requested, T)
    nsteps = ceil(Int, T/dt_requested)
    dt = T/nsteps
    u = heat_explicit_steps(u0, α, dx, dt, nsteps)
    return u, dt, α*dt/dx^2
end

T = 0.02
let previous_error = nothing
    for nx_test in (21, 41, 81, 161)
        xtest = collect(range(0.0, 1.0, length=nx_test))
        dxtest = xtest[2]-xtest[1]
        initial = sin.(π*xtest)
        numerical, dt_used, μ_used = explicit_to_time(initial, α, dxtest, 0.4dxtest^2/α, T)
        exact = exp(-α*π^2*T).*sin.(π*xtest)
        error = maximum(abs.(numerical-exact))
        order = isnothing(previous_error) ? "---" : string(round(log2(previous_error/error), digits=3))
        println("nx=$nx_test: dt=$(round(dt_used,sigdigits=4)), μ=$(round(μ_used,digits=4)), " *
                "最大誤差=$(round(error,sigdigits=5)), 観測次数=$order")
        previous_error = error
    end
end


nx=21: dt=0.001, μ=0.4, 最大誤差=0.00046856, 観測次数=---
nx=41: dt=0.00025, μ=0.4, 最大誤差=0.00011674, 観測次数=2.005
nx=81: dt=6.25e-5, μ=0.4, 最大誤差=2.916e-5, 観測次数=2.001
nx=161: dt=1.562e-5, μ=0.4, 最大誤差=7.2885e-6, 観測次数=2.0


## 後退Euler法：安定性と線形方程式

前進Euler法は現在の差分を使った。**後退Euler法**では、次の時刻の差分を使う。次の状態が式の両辺に現れるので、その状態を方程式の解として求める。

半離散化した連立ODE

$$
v'=\frac{\alpha}{\Delta x^2}Lv
$$

へ後退Euler法を適用すると、

$$
\frac{v^{n+1}-v^n}{\Delta t}
=\frac{\alpha}{\Delta x^2}Lv^{n+1}
$$

より

$$
(I-\mu L)v^{n+1}=v^n
$$

を得る。拡散方程式に対して後退Euler法は刻み幅によらず安定である。ただし、$\Delta t$ を大きくしても時間離散化誤差は $O(\Delta t)$ のままであり、安定性と精度は別である。

本例のように拡散係数・格子・時間刻みが一定なら、係数行列 $I-\mu L$ は時間stepの間で変わらない。最初に一度だけ分解し、その分解を各stepで再利用する。逆行列を陽に作らず、疎な三重対角構造を利用して線形方程式を解く。これは第4章の計算量と第5章のLU分解で学んだ方針そのものである。


In [6]:
function heat_backward_euler(v0, L, α, dx, dt_requested, T)
    nsteps = ceil(Int, T/dt_requested)
    dt = T/nsteps
    μ = α*dt/dx^2
    factorization = factorize(I-μ*L)  # 一度だけ分解する
    v = float.(copy(v0))
    for _ in 1:nsteps
        v = factorization \ v
    end
    return v, dt, μ
end

nx_imp = 321  # 空間誤差が時間誤差を隠さないよう、細かい格子を用いる
x_imp = collect(range(0.0, 1.0, length=nx_imp))
dx_imp = x_imp[2]-x_imp[1]
L_imp = laplacian_1d(nx_imp-2)
v0_imp = sin.(π*x_imp[2:end-1])

for dt_requested in (0.01, 0.005, 0.0025)
    v, dt_used, μ_used = heat_backward_euler(v0_imp, L_imp, α, dx_imp, dt_requested, T)
    exact = exp(-α*π^2*T).*sin.(π*x_imp[2:end-1])
    println("dt=$dt_used, μ=$(round(μ_used,digits=3)): 最大誤差=$(maximum(abs.(v-exact)))")
end


dt=0.01, μ=1024.0: 最大誤差=0.007541611349398791
dt=0.005, μ=512.0: 最大誤差=0.0038815466623715045
dt=0.0025, μ=256.0: 最大誤差=0.001970347406788364


## 精度・安定性・計算量を分けて比較する

内部点数を約2倍にすると、陽解法の1ステップの仕事量は約2倍になる。しかし同じ最終時刻まで進むには、安定条件のためステップ数が約4倍必要となり、総仕事量は約8倍になる。**1回の更新と、計算全体の費用を区別する**ことが大切である。

本例の三重対角構造を使えば、陰解法の分解と各求解も $O(N)$ で行える。これは一般の密行列のLU分解の $O(N^3)$ とは異なる。ただし、陰解法でも精度のために時間刻みを小さくする必要はある。

検証では、初期値・境界値・最終時刻をそろえ、格子幅と時間刻みを記録する。本文の陽解法の細分化試験は、空間と時間を同時に細かくした総誤差の試験である。どちらの誤差が支配するかを調べるなら、一方を十分小さくして他方だけを変える。境界条件が変われば保存する量も変わるため、例えば本例の零Dirichlet境界で熱の総量の保存を要求してはいけない。


## $\clubsuit$ Fourierモードによる安定性解析

[第13章「Fourier解析」](Chap_FourierAnalysis.ipynb)で扱う考え方を使い、無限格子または周期境界上の誤差を波

$$
e_j^n=G^n e^{\mathrm{i}j\theta}
$$

の重ね合わせとして考える。陽解法の更新式へ代入すると、1 stepの増幅率は

$$
G(\theta)=1+\mu(e^{-\mathrm{i}\theta}-2+e^{\mathrm{i}\theta})
=1-4\mu\sin^2\frac{\theta}{2}
$$

となる。すべての波数 $\theta$ について $|G(\theta)|\leq1$ であれば、どの誤差成分も増幅されない。最も厳しいのは $\theta=\pi$ の格子点ごとに符号が変わるモードで、

$$
|1-4\mu|\leq1
$$

より $0\leq\mu\leq1/2$ を得る。本文の零Dirichlet境界では許される波の形が異なるが、この条件はやはり十分条件となる。本文で細かな振動を加えた理由も、同じ増幅の考え方から分かる。

行列の観点では、$L$ の固有ベクトルが離散的な波のモードに対応し、各固有値に対してODEのEuler安定条件を課している。この接続は[第11章「固有値問題と反復法」](Chap_EigenvalueProblems.ipynb)で扱う。


## $\clubsuit$ Crank--Nicolson法

前進Euler法は時刻 $t_n$ の空間差分を、後退Euler法は $t_{n+1}$ の空間差分を使う。両者を平均するとCrank--Nicolson法

$$
\frac{v^{n+1}-v^n}{\Delta t}
=\frac{\alpha}{2\Delta x^2}L(v^n+v^{n+1})
$$

が得られる。整理すれば

$$
\left(I-\frac{\mu}{2}L\right)v^{n+1}
=\left(I+\frac{\mu}{2}L\right)v^n
$$

である。空間2次、時間2次で、拡散方程式に対して刻み幅によらず安定である。ただし非常に大きな $\mu$ では高周波成分が符号を変えながらゆっくり減衰し、非物理的な振動が見えることがある。A安定であっても、すべての不要成分を強く減衰させるとは限らない。


In [7]:
function heat_crank_nicolson(v0, L, α, dx, dt_requested, T)
    nsteps = ceil(Int, T/dt_requested)
    dt = T/nsteps
    μ = α*dt/dx^2
    left = factorize(I-(μ/2)*L)
    right = I+(μ/2)*L
    v = float.(copy(v0))
    for _ in 1:nsteps
        v = left \ (right*v)
    end
    return v, dt
end

for dt_requested in (0.01, 0.005, 0.0025)
    v, dt_used = heat_crank_nicolson(v0_imp, L_imp, α, dx_imp, dt_requested, T)
    exact = exp(-α*π^2*T).*sin.(π*x_imp[2:end-1])
    error = maximum(abs.(v-exact))
    println("dt=$dt_used: Crank--Nicolson最大誤差=$error")
end


dt=0.01: Crank--Nicolson最大誤差=0.00013040660330321252
dt=0.005: Crank--Nicolson最大誤差=3.159144964981486e-5
dt=0.0025: Crank--Nicolson最大誤差=6.919659981452497e-6


## $\clubsuit$ 境界条件と保存量

境界条件は、内部点の差分式へ付け足す小さな修正ではなく、連続問題の一部である。非零Dirichlet条件

$$
u(0,t)=g_0(t),\qquad u(1,t)=g_1(t)
$$

では、最初と最後の内部点の式に境界値が既知の右辺として入る。時間依存なら毎step更新する必要がある。

両端のNeumann条件 $u_x=0$ を有限体積的に離散化すると、離散Laplacianの端の行は概念的に

$$
[-1,1,0,\ldots],\qquad [\ldots,0,1,-1]
$$

となる。この対称な行列は定数ベクトルを零空間に持つ。行和だけでなく列和も0なので、全成分の和、すなわち等しいセル体積で測った離散的な空間平均を保存する。格子点を境界上に置く別の離散化では、端点に半分の重みを付けるなど、保存量の定義も対応させる。Dirichlet条件の行列とは固有値や長時間挙動が異なる。

境界差分の次数が内部差分より低いと、計算全体の収束次数を下げることがある。境界条件を実装したら、境界値または流束そのものに加えて、期待される保存量や散逸量も検証する。


## まとめ

`````{admonition} 本章のまとめ
:class: tip

拡散方程式は、空間の2階中心差分によって連立ODEとなり、時間更新によって計算できる。初期条件と境界条件は問題の一部であり、式だけを離散化しても計算は定まらない。

陽Euler法では近傍値の平均という解釈から刻みの制約が分かる。後退Euler法では線形方程式を解き、一定の係数行列の分解を再利用する。どちらも、安定性と精度を分けて確かめる。

`````

本ノートは、第8章の差分、第9章のODE、第5章の連立一次方程式を組み合わせる補足資料である。[第11章「固有値問題と反復法」](Chap_EigenvalueProblems.ipynb)以降では、行列の固有方向・低次元表現・確率的な計算へ応用を広げる。

## 演習

本文の式・表・出力を使って説明してよい。

`````{admonition} 演習
:class: attention

1. **式から更新を読む**：内部点が3個の場合の行列を本文で確認し、中央の行がどの3点を結んでいるか説明せよ。近傍値 $(1,2,1)$、$\mu=0.4$ で中央の次の値を求めよ。
2. **細かくすると何が増えるか**：空間刻みを半分にし、$\mu$ を同じ値に保つ。時間刻み、同じ最終時刻までのステップ数、陽解法の総仕事量は、それぞれどう変わるか。
3. **安定でも検証する**：後退Euler法で大きな刻みを使い、計算は発散せずに終了した。精度を確かめるには何と比較するか。また、この章の境界条件で温度の総量が減ることは、直ちに実装の誤りと言えるか。

`````



## 参考文献と関連する節

- Driscoll and Braun, *Fundamentals of Numerical Computation*, [Diffusion equations](https://tobydriscoll.net/fnc-julia/diffusion/overview.html).
